In [ ]:
import torch
import math

print("=== PISCES CONCEPT VECTOR UNLEARNING (HOOK CLEANUP & FIXED) ===")

TARGET_LAYER = 12

# 1. CLEANUP ALL PREVIOUS STALE HOOKS FROM MODEL LAYERS
for layer in model.model.layers:
    layer._forward_hooks.clear()

print("✅ Cleared all stale layer hooks!")

# 2. Prompts for Concept Vector Extraction
pos_prompts = [
    "Harry Potter is a wizard at Hogwarts.",
    "Voldemort created Horcruxes to achieve immortality.",
    "Hermione Granger and Ron Weasley are Harry's best friends."
]

neg_prompts = [
    "Photosynthesis converts sunlight into chemical energy.",
    "The capital city of France is Paris.",
    "Albert Einstein developed the theory of relativity."
]

# 3. Extract Concept Vector safely
def get_layer_activations(prompts, layer_idx=12):
    acts = []
    input_device = model.model.embed_tokens.weight.device
    for p in prompts:
        inputs = tokenizer(p, return_tensors="pt").to(input_device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            # Move activation to CPU immediately to avoid multi-GPU device mismatch
            hidden = outputs.hidden_states[layer_idx].mean(dim=1).cpu()
            acts.append(hidden)
    return torch.stack(acts).mean(dim=0)

print("Extracting Concept Vector from Layer 12...")
vec_pos = get_layer_activations(pos_prompts, TARGET_LAYER)
vec_neg = get_layer_activations(neg_prompts, TARGET_LAYER)

concept_vec = vec_pos - vec_neg
concept_vec = concept_vec / torch.norm(concept_vec) # Normalize

print("✅ Concept Vector Extracted Successfully!")

# 4. Helper Function for Perplexity
def compute_ppl(model, tokenizer, text):
    input_device = model.model.embed_tokens.weight.device
    inputs = tokenizer(text, return_tensors="pt").to(input_device)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        return math.exp(outputs.loss.item())

forget_text = "Voldemort used dark magic to create Horcruxes in Harry Potter."
retain_text = "Photosynthesis is the process used by plants to convert light energy."

# Baseline PPL
base_forget = compute_ppl(model, tokenizer, forget_text)
base_retain = compute_ppl(model, tokenizer, retain_text)

# 5. Apply Directional Steering Hook
gamma = 2.5
layer_module = model.model.layers[TARGET_LAYER]

def make_vector_hook(vec, g):
    def hook(module, input, output):
        hidden = output[0] if isinstance(output, tuple) else output
        
        # Dynamically align vector device and dtype with hidden tensor
        v = vec.to(device=hidden.device, dtype=hidden.dtype)
        if v.ndim == 1:
            v = v.unsqueeze(0)
            
        proj = torch.matmul(hidden, v.T)
        modified = hidden - g * torch.matmul(proj, v)
        
        if isinstance(output, tuple):
            return (modified,) + output[1:]
        return modified
    return hook

# Register new hook cleanly
handle = layer_module.register_forward_hook(make_vector_hook(concept_vec, gamma))

try:
    vec_forget_ppl = compute_ppl(model, tokenizer, forget_text)
    vec_retain_ppl = compute_ppl(model, tokenizer, retain_text)

    # Calculate CIS Score
    f_delta = (vec_forget_ppl - base_forget) / base_forget
    r_delta = abs(vec_retain_ppl - base_retain) / base_retain
    vec_cis = f_delta - r_delta

    print("\n================ PISCES VECTOR RESULTS ================")
    print(f"Baseline Forget PPL : {base_forget:.2f}  | Retain PPL: {base_retain:.2f}")
    print(f"PISCES Vector Forget: {vec_forget_ppl:.2f} | Retain PPL: {vec_retain_ppl:.2f}")
    print(f"Calculated CIS Score: {vec_cis:+.4f}")
    print("=======================================================")

finally:
    # Ensure hook removal even if calculation fails
    handle.remove()
    print("✅ Hook removed safely.")

In [ ]:
import os, torch, gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("GPU available:", torch.cuda.is_available())

if not os.path.exists("PISCES"):
    os.system("git clone https://github.com/yoavgur/PISCES.git")

os.system("pip install transformer_lens sae_lens dataclasses_json -q")

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=hf_token)
print("Login successful!")
print("=== SETUP COMPLETE ===")


In [ ]:
import sys
sys.path.append("/kaggle/working/PISCES")

import torch
from abc import ABC, abstractmethod

# --- এই দুইটা class সরাসরি evals.py থেকে কপি করা (broken gcg_multiple import এড়াতে) ---
class AbstractModel(ABC):
    def __init__(self, model, it=True):
        self.model = model
        self.it = it
    def is_it(self):
        return self.it
    def wrap_prompt(self, prompt):
        return prompt

class TransformerLensModel(AbstractModel):
    def generate(self, prompt, max_new_tokens=50, temperature=0.1, do_sample=False):
        resp = self.model.generate(prompt, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=do_sample, verbose=False)
        if self.is_it():
            if "gemma" in self.tokenizer_name().lower():
                index = resp.find("model")
                return resp[index + len("model"):].strip()
        return resp

    def tokenizer_name(self):
        return self.model.cfg.tokenizer_name

    def to_single_token(self, letter):
        return self.model.to_single_token(letter)

    def forward(self, prompt):
        if not isinstance(prompt, torch.Tensor):
            prompt = self.model.to_tokens(prompt)
        return self.model(prompt)

    def to_str_tokens(self, tokens):
        return self.model.to_str_tokens(tokens)

    def to_tokens(self, prompts):
        return self.model.to_tokens(prompts)

    def wrap_prompt(self, prompt):
        if self.is_it():
            s = self.model.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
            return s[5:] if "gemma" in self.tokenizer_name().lower() else s
        return prompt

    def get_pad_token_id(self):
        return self.model.tokenizer.pad_token_id

print("TransformerLensModel defined (bypassing broken evals.py import)")


In [ ]:
from transformer_lens import HookedTransformer
from editor import unlearn_concept, Feature, Concept, get_mlp_act_signs

gc.collect(); torch.cuda.empty_cache()

model = HookedTransformer.from_pretrained("google/gemma-2-2b-it", dtype=torch.float32)
tm = TransformerLensModel(model)
print("Model loaded via transformer_lens!")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer
from editor import unlearn_concept, Feature, Concept, get_mlp_act_signs

gc.collect(); torch.cuda.empty_cache()

local_path = "/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2"

print("Loading HF model from local Kaggle path...")
hf_model = AutoModelForCausalLM.from_pretrained(local_path, torch_dtype=torch.float32)
hf_tokenizer = AutoTokenizer.from_pretrained(local_path)

print("Wrapping into HookedTransformer (no re-download needed)...")
model = HookedTransformer.from_pretrained(
    "google/gemma-2-2b-it",
    hf_model=hf_model,
    tokenizer=hf_tokenizer,
    dtype=torch.float32,
)

tm = TransformerLensModel(model)
print("Model loaded successfully via transformer_lens (local weights)!")


In [ ]:
gc.collect(); torch.cuda.empty_cache()

model = HookedTransformer.from_pretrained(
    "google/gemma-2-2b-it",
    hf_model=hf_model,
    tokenizer=hf_tokenizer,
    dtype=torch.float32,
    move_to_device=False,   # প্রথমে CPU-তে build করুন, GPU-তে পরে move করুন
)
print("Structure built, moving to GPU...")
model = model.to("cuda")
tm = TransformerLensModel(model)
print("Model loaded successfully!")



In [ ]:
import os, torch, gc, sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("GPU available:", torch.cuda.is_available())

if not os.path.exists("PISCES"):
    os.system("git clone https://github.com/yoavgur/PISCES.git")
os.system("pip install transformer_lens sae_lens dataclasses_json -q")

sys.path.append("/kaggle/working/PISCES")

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=hf_token)

# --- TransformerLensModel (evals.py এড়িয়ে) ---
from abc import ABC, abstractmethod

class AbstractModel(ABC):
    def __init__(self, model, it=True):
        self.model = model
        self.it = it
    def is_it(self):
        return self.it
    def wrap_prompt(self, prompt):
        return prompt

class TransformerLensModel(AbstractModel):
    def generate(self, prompt, max_new_tokens=50, temperature=0.1, do_sample=False):
        resp = self.model.generate(prompt, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=do_sample, verbose=False)
        if self.is_it():
            if "gemma" in self.tokenizer_name().lower():
                index = resp.find("model")
                return resp[index + len("model"):].strip()
        return resp
    def tokenizer_name(self):
        return self.model.cfg.tokenizer_name
    def to_single_token(self, letter):
        return self.model.to_single_token(letter)
    def forward(self, prompt):
        if not isinstance(prompt, torch.Tensor):
            prompt = self.model.to_tokens(prompt)
        return self.model(prompt)
    def to_str_tokens(self, tokens):
        return self.model.to_str_tokens(tokens)
    def to_tokens(self, prompts):



In [ ]:
import os, torch, gc, sys

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("GPU available:", torch.cuda.is_available())

if not os.path.exists("PISCES"):
    os.system("git clone https://github.com/yoavgur/PISCES.git")
os.system("pip install transformer_lens sae_lens dataclasses_json -q")

sys.path.append("/kaggle/working/PISCES")

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=hf_token)

# --- TransformerLensModel (evals.py এড়িয়ে) ---
from abc import ABC, abstractmethod

class AbstractModel(ABC):
    def __init__(self, model, it=True):
        self.model = model
        self.it = it
    def is_it(self):
        return self.it
    def wrap_prompt(self, prompt):
        return prompt

class TransformerLensModel(AbstractModel):
    def generate(self, prompt, max_new_tokens=50, temperature=0.1, do_sample=False):
        resp = self.model.generate(prompt, max_new_tokens=max_new_tokens, temperature=temperature, do_sample=do_sample, verbose=False)
        if self.is_it():
            if "gemma" in self.tokenizer_name().lower():
                index = resp.find("model")
                return resp[index + len("model"):].strip()
        return resp
    def tokenizer_name(self):
        return self.model.cfg.tokenizer_name
    def to_single_token(self, letter):
        return self.model.to_single_token(letter)
    def forward(self, prompt):
        if not isinstance(prompt, torch.Tensor):
            prompt = self.model.to_tokens(prompt)
        return self.model(prompt)
    def to_str_tokens(self, tokens):
        return self.model.to_str_tokens(tokens)
    def to_tokens(self, prompts):
        return self.model.to_tokens(prompts)
    def wrap_prompt(self, prompt):
        if self.is_it():
            s = self.model.tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
            return s[5:] if "gemma" in self.tokenizer_name().lower() else s
        return prompt
    def get_pad_token_id(self):
        return self.model.tokenizer.pad_token_id

print("=== SETUP COMPLETE ===")



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer
from editor import unlearn_concept, Feature, Concept, get_mlp_act_signs

gc.collect(); torch.cuda.empty_cache()

local_path = "/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2"

print("Loading HF model from local path...")
hf_model = AutoModelForCausalLM.from_pretrained(local_path, torch_dtype=torch.float32)
hf_tokenizer = AutoTokenizer.from_pretrained(local_path)

print("Building HookedTransformer (CPU first, then move to GPU)...")
model = HookedTransformer.from_pretrained(
    "google/gemma-2-2b-it",
    hf_model=hf_model,
    tokenizer=hf_tokenizer,
    dtype=torch.float32,
    move_to_device=False,
)
print("Structure built, moving to GPU...")
model = model.to("cuda")
tm = TransformerLensModel(model)
print("Model loaded successfully!")



In [1]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer
from editor import unlearn_concept, Feature, Concept, get_mlp_act_signs

local_path = "/kaggle/input/models/google/gemma-2/transformers/gemma-2-2b-it/2"

print("Loading HF model (float16 — memory efficient)...")
hf_model = AutoModelForCausalLM.from_pretrained(
    local_path,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
hf_tokenizer = AutoTokenizer.from_pretrained(local_path)

print("Building HookedTransformer (CPU first)...")
model = HookedTransformer.from_pretrained(
    "google/gemma-2-2b-it",
    hf_model=hf_model,
    tokenizer=hf_tokenizer,
    dtype=torch.float16,
    move_to_device=False,
)

del hf_model
gc.collect(); torch.cuda.empty_cache()

print("Moving to GPU...")
model = model.to("cuda")
tm = TransformerLensModel(model)
print("Model loaded successfully!")



ModuleNotFoundError: No module named 'editor'